In [ ]:
from pathlib import Path
import pandas as pd

# Local NeuroShell project path: supports running from project root,
# notebooks/, or data/processed/ without hard-coded Windows/Colab paths.
project_root = next(
    (
        parent for parent in (Path.cwd(), *list(Path.cwd().parents)[:2])
        if (parent / "data" / "processed" / "NeuroShell_C4_train.csv").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "NeuroShell_C4_train.csv not found. Open the notebook from within "
        "your Neuroshell project and keep the CSVs in data/processed/."
    )

data_dir = project_root / "data" / "processed"
train_path = data_dir / "NeuroShell_C4_train.csv"
val_path = data_dir / "CCEE_validation.csv"
test_path = data_dir / "CCEE_test.csv"

# Verify all three input datasets exist before training.
for path in (train_path, val_path, test_path):
    assert path.is_file(), f"Missing dataset: {path}"

# Load datasets
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

# Guard against accidentally loading a different training experiment.
expected_rows = {"Training": 150852, "Validation": 24937, "Testing": 31171}
for name, df in (("Training", train_df), ("Validation", val_df), ("Testing", test_df)):
    expected = expected_rows[name]
    if len(df) != expected:
        raise ValueError(
            f"{name} dataset has {len(df):,} rows; expected {expected:,}. "
            "Check that the CSV belongs to the final XGBoost training split."
        )

print("===== DATASET INFORMATION =====")

for name, df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(f"\n{name} Dataset")
    print("Total Records:", len(df))
    print("Total Features/Columns:", len(df.columns))
    print("Class Distribution:")
    print(df["label"].value_counts())

# Check dataset separation
assert set(train_df["cve_id"]).isdisjoint(
    set(val_df["cve_id"])
)
assert set(train_df["cve_id"]).isdisjoint(
    set(test_df["cve_id"])
)
assert set(val_df["cve_id"]).isdisjoint(
    set(test_df["cve_id"])
)

print("\n✅ STEP 1 COMPLETED!")


In [ ]:

import pandas as pd
import numpy as np

# Five features used for XGBoost classification
features = [
    "base_score",
    "epss_score",
    "epss_perc",
    "age_days",
    "cvss_epss_product"
]

target = "label"

# Prepare feature matrices and target variables
X_train = train_df[features].copy()
y_train = train_df[target].astype(int)

X_val = val_df[features].copy()
y_val = val_df[target].astype(int)

X_test = test_df[features].copy()
y_test = test_df[target].astype(int)

# Validate numerical features
for name, X, y in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Testing", X_test, y_test)
]:
    assert X.notna().all().all(), f"{name}: Missing values"
    assert np.isfinite(X.to_numpy(dtype=float)).all()
    assert y.isin([0, 1]).all()

print("===== SELECTED FEATURES =====")
for i, feature in enumerate(features, 1):
    print(f"{i}. {feature}")

print("\n===== DATA VALIDATION =====")
print("Training Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Testing Shape:", X_test.shape)

print("\nTraining Missing Values:", X_train.isna().sum().sum())
print("Validation Missing Values:", X_val.isna().sum().sum())
print("Testing Missing Values:", X_test.isna().sum().sum())

print("\n===== RESAMPLING CHECK =====")
print(
    "Training Unique CVEs:",
    train_df["cve_id"].nunique()
)
print(
    "Training Repeated Rows:",
    train_df["cve_id"].duplicated().sum()
)

print("\n✅ STEP 2 COMPLETED!")


In [ ]:

import time
import xgboost as xgb
from xgboost import XGBClassifier

# XGBoost model configuration
model = XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=1,
    random_state=42,
    n_jobs=-1
)

print("===== XGBOOST MODEL CONFIGURATION =====")
print("Algorithm: XGBoost Classifier")
print("Number of Features:", X_train.shape[1])
print("Training Samples:", len(X_train))
print("Estimators:", model.n_estimators)
print("Maximum Depth:", model.max_depth)
print("Learning Rate:", model.learning_rate)

print("\n===== MODEL TRAINING STARTED =====")

start_time = time.time()

model.fit(X_train, y_train)

training_time = time.time() - start_time

print("\n✅ MODEL TRAINING COMPLETED!")
print(f"Training Time: {training_time:.2f} seconds")

# Save trained model
# Retraining output is separate, so the deployed final model is not overwritten.
model_path = project_root / "models" / "xgboost_150852_retrained_check.json"
model_path.parent.mkdir(parents=True, exist_ok=True)
model.save_model(model_path)

print("\n===== SAVED MODEL =====")
print("Model:", model_path)
print("XGBoost Version:", xgb.__version__)

print("\n✅ STEP 3 COMPLETED!")


In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# STEP 4.1: Predictions on validation dataset
y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)[:, 1]

# STEP 4.2: Calculate metrics
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(
    y_val, y_val_pred, zero_division=0
)
recall = recall_score(
    y_val, y_val_pred, zero_division=0
)
f1 = f1_score(
    y_val, y_val_pred, zero_division=0
)
roc_auc = roc_auc_score(y_val, y_val_prob)
pr_auc = average_precision_score(y_val, y_val_prob)

print("===== XGBOOST VALIDATION RESULTS =====")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print(f"ROC-AUC   : {roc_auc * 100:.2f}%")
print(f"PR-AUC    : {pr_auc * 100:.2f}%")

# STEP 4.3: Confusion Matrix
cm = confusion_matrix(
    y_val, y_val_pred, labels=[0, 1]
)

print("\n===== CONFUSION MATRIX =====")
print(cm)

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    y_val, y_val_pred,
    zero_division=0,
    digits=4
))

# STEP 4.4: Majority-class baseline
baseline_accuracy = (y_val == 0).mean()

print("\n===== BASELINE COMPARISON =====")
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print(f"Always Class 0: {baseline_accuracy * 100:.2f}%")

# STEP 4.5: Confusion Matrix Visualization
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Class 0", "Class 1"]
)

fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d")

ax.set_title("XGBoost Validation Confusion Matrix")
plt.tight_layout()

plt.savefig(
    project_root / "models" / "xgboost_validation_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n✅ STEP 4 COMPLETED!")


In [ ]:
# Local notebook: the newly trained model is already saved on disk.
print("Retrained model output:", model_path.resolve())
print("Production model remains unchanged:", (project_root / "models" / "xgboost_150852_custom.json").resolve())
